# Compton Wavelength Stopping Criterium

## Imports

In [68]:
# Important modules
import os
import sys
import subprocess
import pickle
import numpy as np

sys.path.append('/home/ws/om2854/jaxionsdir/jaxions/scripts')
os.environ['JAXIONS_DIR'] = '/home/ws/om2854/jaxionsdir'

# Get PATH from default login shell
shell_path = subprocess.check_output(
    ['bash', '-l', '-c', 'echo $PATH'],
    text = True
).strip()

# Prepend the directory containing vaxion3d to PATH
vaxion_dir = '/home/ws/om2854/jaxionsdir/build/test'
os.environ['PATH'] = f"{vaxion_dir}:{shell_path}"

# Run simulations
from pyaxions import simgen as sg

# Deal with measurement files from simulations
from pyaxions import jaxions as pa

# Kinetic misalignment functions
from kin_mis_utils import genspec_kinetic_mis

## Dictionary Helpers

In [2]:
SAVE_FILE = os.path.join(os.getcwd(), "compton_cosmos.pkl")

def load_results():
    """Load existing results dictionary from pickle file, or create new one."""
    if os.path.exists(SAVE_FILE):
        with open(SAVE_FILE, "rb") as f:
            results = pickle.load(f)
    else:
        results = {}
    return results

def save_results(results):
    """Save dictionary back to pickle file."""
    with open(SAVE_FILE, "wb") as f:
        pickle.dump(results, f)

## Simulation Setup

In [60]:
theta1, vheta1 = 0, 0  # Initial field value and velocity value in H1 units of zero mode
N, L, msa, tauf = 64, 3, 1.0, 5.0

# Create initial conditions
genspec_kinetic_mis(N, L, theta1, vheta1, hom = True);

In [61]:
fAGeV = 5e+12 # Change accordingly

## Run Simulation

In [62]:
# Run simulations
rank, jax = sg.simgen(N = N, L = L, msa = msa, zRANKS = 1, ctf = tauf, ic = 'spax', ict = 'spax', prec = 'single',
                      dev = 'cpu', lap = 2, fftplan = 64, fA = fAGeV, vqcd = 'vqcdC', xtr = ' --ftype axion --mode0 1',
                      cti = 1, meas = 1, spKGV = 15, rmask = 4.0, p3D = 0, p2Dmap = False, p2DmapE = False, p2DmapPE = False, 
                      slc = N//2, nologmpi = True, verbose = 3, sIter = 5, verb = False, dump = 10, wDz = 0.2)
    
sg.runsim(JAX = jax, RANK = rank, THR = 1//rank, USA = '', VERB = False, BONDEN = False)
    
os.system('mv axion.log.0 log-run.txt initialspectrum.dat out')



--------------------------------------------------------------------------------------------
Mode: run 

Overview: N=64, MPI_RANKS=1, L=3.000000, msa=1.000000

Done!
--------------------------------------------------------------------------------------------


0

## Collect Results and Save

In [63]:
# Load measurement file
path = os.getcwd()
file_path = "/out"
mf = pa.findmfiles(path +  file_path)

In [64]:
tautab = pa.gml(mf, 'ct') # Conformal times 
R = pa.gml(mf, 'R') # Scale factor in R₁
mA = pa.gml(mf,'massA') # Axion mass, conformal mass: ma*r

In [65]:
results = load_results()
results[fAGeV] = {"tautab": tautab, "R": R, "mA": mA}
save_results(results)

print(f"Results for fA = {fAGeV} saved in {SAVE_FILE}")

Results for fA = 5000000000000.0 saved in compton_cosmos.pkl


## Dictionary

In [66]:
SAVE_FILE = "compton_cosmos.pkl"

with open(SAVE_FILE, "rb") as f:
    results = pickle.load(f)

print("Available keys in dictionary:")
for key in results.keys():
    print(f"{key:.2e}")

Available keys in dictionary:
1.00e+09
5.00e+09
1.00e+10
5.00e+10
1.00e+11
5.00e+11
1.00e+12
5.00e+12


## Criterium

In [82]:
def compton_stopping_crit(fAGeV, N, L, num_crit = 1.0):
    """
    Determine the conformal time when the number of Compton wavelengths between two grid points
    increases above the threshold num_crit.
    
    Args:
        fAGeV (float): Axion decay constant in GeV.
        N (int): Number of points per spatial dimension.
        L (float): Physical size of simulation box in L1 units.
        num_crit (float): Threshold of number of Compton wavelengths. Defaults to 1.0.
    
    Returns:
        tau_crit (float): Conformal time when number of Compton wavelengths increases above threshold.
    """
    
    # Load precomputed results
    SAVE_FILE = "compton_cosmos.pkl"
    with open(SAVE_FILE, "rb") as f:
        results = pickle.load(f)

    # Find the closest available key
    closest_key = min(results.keys(), key = lambda k: abs(k - fAGeV))
    values = results[closest_key]

    # Compute number of Compton wavelengths between two grid points at each time step
    num_compton = (L/N) * values['R'] * values['mA']

    # Find the first index where num_compton > num_crit
    idx = np.argmax(num_compton > num_crit)
    if num_compton[idx] <= num_crit: # if never exceeded
        return None

    # Corresponding conformal time
    if idx == 0:
        tau_crit = values['tautab'][0]
    else:
        tau_crit = values['tautab'][idx - 1]

    return tau_crit

In [94]:
compton_stopping_crit(fAGeV = 1e9, N = 1024, L = 1, num_crit = 1.0)

4.0159678245724315